<a href="https://colab.research.google.com/github/luissanchezmanas-hash/Ejercicios_Pontia/blob/Entrega_Ej1_Visual/Reporte_PontiaSuperStore_descuentos_CORREGIDO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Estudio de política de descuentos — Pontia SuperStore

**Objetivo:** analizar cómo ha impactado la política de descuentos en la rentabilidad de Pontia SuperStore usando `pandas` para el procesamiento de datos y `plotly.express` para visualizaciones interactivas.

El reporte sigue una estructura narrativa:

1. **Introducción:** ¿existe un problema general de rentabilidad asociado al descuento?
2. **Nudo:** ¿el impacto es homogéneo por nivel de descuento, categoría y subcategoría?
3. **Desenlace:** propuesta de actuación basada en los segmentos donde el descuento destruye margen.

> Nota metodológica: este análisis identifica asociación entre descuento y rentabilidad, no causalidad pura. Para afirmar causalidad haría falta controlar por precio base, elasticidad, región, cliente, campaña, mix de producto y temporalidad.

## 1. Importación de librerías y carga de datos

Se usa únicamente la hoja `Orders`, aunque el fichero tenga también `Returns` y `People`.

Si el entorno no puede leer `.xls`, instala previamente `xlrd`:

```python
pip install xlrd
```

In [ ]:
!pip install xlrd -q

# Procesamiento de datos
import pandas as pd
import numpy as np

# Visualización interactiva
import plotly.express as px

# Configuración visual de pandas
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

# Subida manual del archivo Excel a Colab
from google.colab import files
uploaded = files.upload()

# Comprobamos el nombre real del archivo subido
list(uploaded.keys())

Saving PontiaSuperStore.xls to PontiaSuperStore.xls


['PontiaSuperStore.xls']

In [ ]:
from pathlib import Path
# Procesamiento de datos
import pandas as pd
import numpy as np

# Visualización interactiva
import plotly.express as px

# Detectamos automáticamente el archivo .xls subido
archivos_xls = list(Path(".").glob("*.xls"))

if len(archivos_xls) == 0:
    raise FileNotFoundError("No se ha subido ningún archivo .xls al entorno de Colab.")

ruta_excel = archivos_xls[0]

print(f"Archivo detectado: {ruta_excel}")

# Carga exclusiva de la hoja Orders
df = pd.read_excel(ruta_excel, sheet_name="Orders")

print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]:,}")

df.head()

Archivo detectado: PontiaSuperStore.xls
Filas: 9,996
Columnas: 21


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.00,41.91
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.00,219.58
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.00,6.87
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.58,5,0.45,-383.03
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.37,2,0.20,2.52


## 2. Validación inicial del dataset

Antes de construir KPIs conviene revisar tipos de datos, nulos y rangos básicos. En un informe empresarial, saltarse esta fase es peligroso: una fecha mal interpretada o un campo numérico leído como texto puede invalidar todo el análisis posterior.

In [ ]:
# Tipos de datos
resumen_tipos = pd.DataFrame({
    "columna": df.columns,
    "tipo": df.dtypes.astype(str).values,
    "nulos": df.isna().sum().values,
    "nulos_%": (df.isna().mean().values * 100).round(2)
})

resumen_tipos

,columna,tipo,nulos,nulos_%
0,Row ID,int64,0,0.00
1,Order ID,object,0,0.00
2,Order Date,datetime64[ns],0,0.00
3,Ship Date,datetime64[ns],0,0.00
4,Ship Mode,object,0,0.00
5,Customer ID,object,0,0.00
6,Customer Name,object,0,0.00
7,Segment,object,0,0.00
8,Country,object,80,0.80
9,City,object,0,0.00


In [ ]:
# Comprobación de rangos numéricos relevantes
variables_numericas = ["Sales", "Quantity", "Discount", "Profit"]
df[variables_numericas].describe().T

,count,mean,std,min,25%,50%,75%,max
Sales,"9,996.00",229.83,623.19,0.44,17.28,54.49,209.94,"22,638.48"
Quantity,"9,996.00",3.79,2.23,1.00,2.00,3.00,5.00,14.00
Discount,"9,996.00",0.16,0.21,0.00,0.00,0.20,0.20,0.80
Profit,"9,996.00",28.66,234.24,"-6,599.98",1.73,8.67,29.36,"8,399.98"


## 3. Generación de variables calculadas obligatorias

Se crean exactamente las columnas exigidas:

- `Sales/Quantity`: ventas por unidad.
- `Profit/Quantity`: beneficio por unidad.
- `flag_descuento`: marca si la línea tuvo descuento.
- `Year`: año extraído de `Order Date`.

Además se define explícitamente un KPI central de rentabilidad:

$$
\text{Margen} = \frac{\text{Profit}}{\text{Sales}}
$$
\]

Usaremos dos formas de margen:

- **Margen ponderado:** `sum(Profit) / sum(Sales)`. Es el más adecuado para visión de negocio agregada porque pondera por volumen de ventas.
- **Margen de línea:** `Profit / Sales` por operación. Es útil para distribuciones, pero puede sobrerrepresentar operaciones pequeñas.

In [ ]:
# Columnas obligatorias
# Se protegen divisiones ante posibles cantidades o ventas cero.
df["Sales/Quantity"] = np.where(df["Quantity"] != 0, df["Sales"] / df["Quantity"], np.nan)
df["Profit/Quantity"] = np.where(df["Quantity"] != 0, df["Profit"] / df["Quantity"], np.nan)
df["flag_descuento"] = np.where(df["Discount"] > 0, "Con Descuento", "Sin Descuento")
df["Year"] = pd.to_datetime(df["Order Date"]).dt.year

# KPI explícito de rentabilidad
df["Margen"] = np.where(df["Sales"] != 0, df["Profit"] / df["Sales"], np.nan)

# Revisión de las columnas creadas
columnas_creadas = ["Sales/Quantity", "Profit/Quantity", "flag_descuento", "Year", "Margen"]
df[columnas_creadas].head()

,Sales/Quantity,Profit/Quantity,flag_descuento,Year,Margen
0,130.98,20.96,Sin Descuento,2016,0.16
1,243.98,73.19,Sin Descuento,2016,0.30
2,7.31,3.44,Sin Descuento,2016,0.47
3,191.52,-76.61,Con Descuento,2015,-0.40
4,11.18,1.26,Con Descuento,2015,0.11


## 4. Vista global: ¿los descuentos están asociados a menor rentabilidad?

Primero se compara el conjunto de operaciones con y sin descuento. Este gráfico presenta el problema: si los descuentos elevan ventas pero reducen beneficio, la política comercial puede estar comprando volumen a costa de margen.

In [ ]:
resumen_descuento = (
    df.groupby("flag_descuento", as_index=False)
      .agg(
          lineas=("Row ID", "count"),
          pedidos=("Order ID", "nunique"),
          ventas=("Sales", "sum"),
          beneficio=("Profit", "sum"),
          cantidad=("Quantity", "sum"),
          descuento_medio=("Discount", "mean"),
          precio_unitario_medio=("Sales/Quantity", "mean"),
          beneficio_unitario_medio=("Profit/Quantity", "mean")
      )
)

resumen_descuento["margen_ponderado"] = resumen_descuento["beneficio"] / resumen_descuento["ventas"]
resumen_descuento["beneficio_por_unidad_agregado"] = resumen_descuento["beneficio"] / resumen_descuento["cantidad"]

resumen_descuento

,flag_descuento,lineas,pedidos,ventas,beneficio,cantidad,descuento_medio,precio_unitario_medio,beneficio_unitario_medio,margen_ponderado,beneficio_por_unidad_agregado
0,Con Descuento,5196,2954,"1,209,292.39","-34,590.58",19606,0.30,62.78,-1.23,-0.03,-1.76
1,Sin Descuento,4800,2644,"1,088,084.78","321,046.54",18279,0.00,58.89,17.57,0.30,17.56


In [ ]:
print(resumen_descuento.shape)
print(resumen_descuento.columns)
display(resumen_descuento)

(2, 11)
Index(['flag_descuento', 'lineas', 'pedidos', 'ventas', 'beneficio',
       'cantidad', 'descuento_medio', 'precio_unitario_medio',
       'beneficio_unitario_medio', 'margen_ponderado',
       'beneficio_por_unidad_agregado'],
      dtype='object')


,flag_descuento,lineas,pedidos,ventas,beneficio,cantidad,descuento_medio,precio_unitario_medio,beneficio_unitario_medio,margen_ponderado,beneficio_por_unidad_agregado
0,Con Descuento,5196,2954,"1,209,292.39","-34,590.58",19606,0.30,62.78,-1.23,-0.03,-1.76
1,Sin Descuento,4800,2644,"1,088,084.78","321,046.54",18279,0.00,58.89,17.57,0.30,17.56


In [ ]:
# Pasamos de formato ancho a formato largo para que Plotly lo interprete correctamente
resumen_descuento_long = resumen_descuento.melt(
    id_vars=[
        "flag_descuento",
        "lineas",
        "pedidos",
        "cantidad",
        "descuento_medio",
        "precio_unitario_medio",
        "beneficio_unitario_medio",
        "margen_ponderado",
        "beneficio_por_unidad_agregado"
    ],
    value_vars=["ventas", "beneficio"],
    var_name="metrica",
    value_name="importe"
)

fig1 = px.bar(
    resumen_descuento_long,
    x="flag_descuento",
    y="importe",
    color="metrica",
    barmode="group",
    text_auto=".2s",
    title="Ventas y beneficio agregado según existencia de descuento",
    labels={
        "flag_descuento": "Tipo de operación",
        "importe": "Importe agregado",
        "metrica": "Métrica"
    },
    hover_data={
        "flag_descuento": True,
        "metrica": True,
        "importe": ":,.2f",
        "margen_ponderado": ":.2%",
        "descuento_medio": ":.2%",
        "lineas": ":,",
        "pedidos": ":,"
    }
)

fig1.update_layout(
    yaxis_tickprefix="$",
    legend_title_text="Métrica"
)

fig1.show()

In [ ]:
ventas_con_desc = resumen_descuento.loc[resumen_descuento["flag_descuento"] == "Con Descuento", "ventas"].iloc[0]
benef_con_desc = resumen_descuento.loc[resumen_descuento["flag_descuento"] == "Con Descuento", "beneficio"].iloc[0]
margen_con_desc = resumen_descuento.loc[resumen_descuento["flag_descuento"] == "Con Descuento", "margen_ponderado"].iloc[0]
ventas_sin_desc = resumen_descuento.loc[resumen_descuento["flag_descuento"] == "Sin Descuento", "ventas"].iloc[0]
benef_sin_desc = resumen_descuento.loc[resumen_descuento["flag_descuento"] == "Sin Descuento", "beneficio"].iloc[0]
margen_sin_desc = resumen_descuento.loc[resumen_descuento["flag_descuento"] == "Sin Descuento", "margen_ponderado"].iloc[0]

print("Conclusión del gráfico 1")
print("- Las operaciones con descuento concentran ventas relevantes, pero su beneficio agregado es negativo.")
print(f"- Con descuento: ventas = {ventas_con_desc:,.2f}, beneficio = {benef_con_desc:,.2f}, margen ponderado = {margen_con_desc:.2%}.")
print(f"- Sin descuento: ventas = {ventas_sin_desc:,.2f}, beneficio = {benef_sin_desc:,.2f}, margen ponderado = {margen_sin_desc:.2%}.")
print("- La hipótesis de trabajo es clara: el descuento no está funcionando como simple acelerador de ventas; en agregado está erosionando rentabilidad.")

Conclusión del gráfico 1
- Las operaciones con descuento concentran ventas relevantes, pero su beneficio agregado es negativo.
- Con descuento: ventas = 1,209,292.39, beneficio = -34,590.58, margen ponderado = -2.86%.
- Sin descuento: ventas = 1,088,084.78, beneficio = 321,046.54, margen ponderado = 29.51%.
- La hipótesis de trabajo es clara: el descuento no está funcionando como simple acelerador de ventas; en agregado está erosionando rentabilidad.


## 5. Relación entre intensidad del descuento y margen

Ahora no basta con distinguir entre descuento sí/no. El riesgo comercial suele aparecer cuando el descuento cruza ciertos umbrales. Por eso se agrupa `Discount` en tramos y se analiza el margen ponderado.

In [ ]:
# Tramos de descuento. Se incluye 0% como grupo propio.
bins = [-0.001, 0, 0.10, 0.20, 0.30, 0.50, 1.00]
labels = ["0%", "0-10%", "10-20%", "20-30%", "30-50%", ">50%"]

df["tramo_descuento"] = pd.cut(df["Discount"], bins=bins, labels=labels)

resumen_tramos = (
    df.groupby("tramo_descuento", observed=False, as_index=False)
      .agg(
          lineas=("Row ID", "count"),
          ventas=("Sales", "sum"),
          beneficio=("Profit", "sum"),
          descuento_medio=("Discount", "mean")
      )
)

resumen_tramos["margen_ponderado"] = resumen_tramos["beneficio"] / resumen_tramos["ventas"]
resumen_tramos

,tramo_descuento,lineas,ventas,beneficio,descuento_medio,margen_ponderado
0,0%,4800,"1,088,084.78","321,046.54",0.00,0.30
1,0-10%,94,"54,369.35","9,029.18",0.10,0.17
2,10-20%,3709,"792,152.89","91,756.30",0.20,0.12
3,20-30%,227,"103,226.65","-10,369.28",0.30,-0.10
4,30-50%,310,"195,314.76","-48,447.73",0.42,-0.25
5,>50%,856,"64,228.74","-76,559.05",0.72,-1.19


In [ ]:
fig2 = px.bar(
    resumen_tramos,
    x="tramo_descuento",
    y="margen_ponderado",
    text="margen_ponderado",
    title="Margen ponderado por tramo de descuento",
    labels={
        "tramo_descuento": "Tramo de descuento",
        "margen_ponderado": "Margen ponderado Profit / Sales"
    },
    hover_data={
        "ventas": ":,.2f",
        "beneficio": ":,.2f",
        "lineas": ":,",
        "descuento_medio": ":.2%",
        "margen_ponderado": ":.2%"
    }
)

fig2.update_traces(texttemplate="%{text:.1%}", textposition="outside")
fig2.update_layout(yaxis_tickformat=".0%")
fig2.add_hline(y=0, line_dash="dash", annotation_text="Umbral de rentabilidad = 0%")
fig2.show()

In [ ]:
print("Conclusión del gráfico 2")
print("- El margen no cae de forma suave: se deteriora claramente al aumentar la intensidad del descuento.")
print("- Los tramos por encima del 20%-30% ya entran en zona de pérdida agregada en este dataset.")
print("- El tramo superior al 50% es especialmente negativo: no debería tratarse como promoción normal, sino como liquidación, error de pricing o venta táctica justificada.")

Conclusión del gráfico 2
- El margen no cae de forma suave: se deteriora claramente al aumentar la intensidad del descuento.
- Los tramos por encima del 20%-30% ya entran en zona de pérdida agregada en este dataset.
- El tramo superior al 50% es especialmente negativo: no debería tratarse como promoción normal, sino como liquidación, error de pricing o venta táctica justificada.


## 6. Análisis por producto: Category y Sub-Category

El descuento no afecta igual a todos los productos. Un error habitual sería prohibir todos los descuentos solo porque el agregado es malo. La decisión correcta requiere segmentación: detectar dónde el descuento mantiene margen y dónde lo destruye.

In [ ]:
resumen_subcategoria = (
    df.groupby(["Category", "Sub-Category", "flag_descuento"], as_index=False)
      .agg(
          lineas=("Row ID", "count"),
          ventas=("Sales", "sum"),
          beneficio=("Profit", "sum"),
          descuento_medio=("Discount", "mean"),
          cantidad=("Quantity", "sum")
      )
)

resumen_subcategoria["margen_ponderado"] = resumen_subcategoria["beneficio"] / resumen_subcategoria["ventas"]
resumen_subcategoria["beneficio_por_unidad"] = resumen_subcategoria["beneficio"] / resumen_subcategoria["cantidad"]

resumen_subcategoria.sort_values("margen_ponderado").head(10)

,Category,Sub-Category,flag_descuento,lineas,ventas,beneficio,descuento_medio,cantidad,margen_ponderado,beneficio_por_unidad
6,Furniture,Tables,Con Descuento,247,"135,386.77","-31,001.78",0.34,931,-0.23,-33.30
30,Technology,Machines,Con Descuento,86,"118,204.63","-23,753.07",0.41,302,-0.20,-78.65
24,Office Supplies,Supplies,Con Descuento,73,"15,114.33","-2,907.49",0.20,247,-0.19,-11.77
8,Office Supplies,Appliances,Con Descuento,195,"29,465.97","-5,045.73",0.40,708,-0.17,-7.13
4,Furniture,Furnishings,Con Descuento,386,"30,255.36","-3,788.83",0.34,1400,-0.13,-2.71
0,Furniture,Bookcases,Con Descuento,168,"82,944.02","-9,548.27",0.29,657,-0.12,-14.53
12,Office Supplies,Binders,Con Descuento,1186,"121,583.25","-9,092.69",0.48,4683,-0.07,-1.94
22,Office Supplies,Storage,Con Descuento,316,"65,989.85","-4,249.35",0.20,1114,-0.06,-3.81
2,Furniture,Chairs,Con Descuento,484,"237,388.37","4,657.07",0.22,1816,0.02,2.56
32,Technology,Phones,Con Descuento,578,"206,127.34","10,150.52",0.24,2187,0.05,4.64


In [ ]:
fig3 = px.treemap(
    resumen_subcategoria,
    path=["Category", "Sub-Category", "flag_descuento"],
    values="ventas",
    color="margen_ponderado",
    color_continuous_midpoint=0,
    title="Mapa de rentabilidad por categoría, subcategoría y descuento",
    labels={
        "ventas": "Ventas",
        "margen_ponderado": "Margen ponderado"
    },
    hover_data={
        "ventas": ":,.2f",
        "beneficio": ":,.2f",
        "margen_ponderado": ":.2%",
        "descuento_medio": ":.2%",
        "lineas": ":,"
    }
)

fig3.show()

In [ ]:
peores_subcategorias_descuento = (
    resumen_subcategoria
    .query("flag_descuento == 'Con Descuento'")
    .sort_values("beneficio")
    .head(10)
    [["Category", "Sub-Category", "ventas", "beneficio", "margen_ponderado", "descuento_medio", "lineas"]]
)

peores_subcategorias_descuento

,Category,Sub-Category,ventas,beneficio,margen_ponderado,descuento_medio,lineas
6,Furniture,Tables,"135,386.77","-31,001.78",-0.23,0.34,247
30,Technology,Machines,"118,204.63","-23,753.07",-0.20,0.41,86
0,Furniture,Bookcases,"82,944.02","-9,548.27",-0.12,0.29,168
12,Office Supplies,Binders,"121,583.25","-9,092.69",-0.07,0.48,1186
8,Office Supplies,Appliances,"29,465.97","-5,045.73",-0.17,0.40,195
22,Office Supplies,Storage,"65,989.85","-4,249.35",-0.06,0.20,316
4,Furniture,Furnishings,"30,255.36","-3,788.83",-0.13,0.34,386
24,Office Supplies,Supplies,"15,114.33","-2,907.49",-0.19,0.20,73
16,Office Supplies,Fasteners,"1,201.44",297.31,0.25,0.20,89
18,Office Supplies,Labels,"3,246.75","1,124.16",0.35,0.20,125


In [ ]:
print("Conclusión del gráfico 3")
print("- El problema no es homogéneo por producto: hay subcategorías donde el descuento sigue siendo rentable y otras donde destruye valor.")
print("- Las subcategorías con descuento que deben revisarse primero son las que combinan volumen relevante, beneficio negativo y margen negativo.")
print("- En el ranking de pérdidas aparecen especialmente Tables, Machines, Bookcases, Binders, Appliances, Storage y Furnishings con descuento.")

Conclusión del gráfico 3
- El problema no es homogéneo por producto: hay subcategorías donde el descuento sigue siendo rentable y otras donde destruye valor.
- Las subcategorías con descuento que deben revisarse primero son las que combinan volumen relevante, beneficio negativo y margen negativo.
- En el ranking de pérdidas aparecen especialmente Tables, Machines, Bookcases, Binders, Appliances, Storage y Furnishings con descuento.


## 7. Evolución temporal: ¿el problema es puntual o recurrente?

La política de descuentos puede tener sentido si responde a un año concreto, una liquidación o un cambio de estrategia. Si el patrón se repite durante varios años, el problema es estructural.

In [ ]:
resumen_anual = (
    df.groupby(["Year", "flag_descuento"], as_index=False)
      .agg(
          ventas=("Sales", "sum"),
          beneficio=("Profit", "sum"),
          lineas=("Row ID", "count")
      )
)

resumen_anual["margen_ponderado"] = resumen_anual["beneficio"] / resumen_anual["ventas"]
resumen_anual

,Year,flag_descuento,ventas,beneficio,lineas,margen_ponderado
0,2014,Con Descuento,"269,287.77","-9,073.19",1056,-0.03
1,2014,Sin Descuento,"214,959.73","58,617.17",937,0.27
2,2015,Con Descuento,"244,012.66","-7,252.15",1079,-0.03
3,2015,Sin Descuento,"226,660.60","68,912.98",1024,0.30
4,2016,Con Descuento,"311,594.86","-7,546.79",1339,-0.02
5,2016,Sin Descuento,"297,646.30","89,358.68",1249,0.30
6,2017,Con Descuento,"384,397.11","-10,718.44",1722,-0.03
7,2017,Sin Descuento,"348,818.15","104,157.71",1590,0.30


In [ ]:
fig4 = px.line(
    resumen_anual,
    x="Year",
    y="margen_ponderado",
    color="flag_descuento",
    markers=True,
    title="Evolución anual del margen ponderado según existencia de descuento",
    labels={
        "Year": "Año",
        "margen_ponderado": "Margen ponderado Profit / Sales",
        "flag_descuento": "Tipo de operación"
    },
    hover_data={
        "ventas": ":,.2f",
        "beneficio": ":,.2f",
        "lineas": ":,",
        "margen_ponderado": ":.2%"
    }
)

fig4.update_layout(yaxis_tickformat=".0%")
fig4.add_hline(y=0, line_dash="dash", annotation_text="Umbral de rentabilidad = 0%")
fig4.show()

In [ ]:
print("Conclusión del gráfico 4")
print("- La comparación anual permite comprobar si la destrucción de margen asociada al descuento es persistente.")
print("- Si las líneas con descuento permanecen cerca o por debajo de cero en varios años, el problema no parece un accidente aislado.")
print("- La recomendación no debe ser solo mirar el total acumulado, sino incorporar control mensual/anual de margen por familia de producto.")

Conclusión del gráfico 4
- La comparación anual permite comprobar si la destrucción de margen asociada al descuento es persistente.
- Si las líneas con descuento permanecen cerca o por debajo de cero en varios años, el problema no parece un accidente aislado.
- La recomendación no debe ser solo mirar el total acumulado, sino incorporar control mensual/anual de margen por familia de producto.


## 8. Profundización: dispersión entre descuento, ventas y beneficio

Este gráfico baja al nivel de línea de pedido. Sirve para detectar operaciones concretas donde un descuento alto coincide con pérdidas severas, y para comprobar si las pérdidas se concentran en pocas operaciones grandes o en muchas operaciones pequeñas.

In [ ]:
df_box = df.copy()

df_box["Discount_pct"] = (df_box["Discount"] * 100).round(0).astype(int).astype(str) + "%"

# Filtrado solo para mejorar visualización
df_box_filtrado = df_box[
    (df_box["Margen"] >= -1) &
    (df_box["Margen"] <= 0.8)
].copy()

fig5_box = px.box(
    df_box_filtrado,
    x="Discount_pct",
    y="Margen",
    color="Category",
    points="outliers",
    title="Distribución del margen según nivel de descuento",
    labels={
        "Discount_pct": "Nivel de descuento",
        "Margen": "Margen Profit / Sales",
        "Category": "Categoría"
    },
    hover_data={
        "Sub-Category": True,
        "Sales": ":,.2f",
        "Profit": ":,.2f",
        "Product Name": True
    }
)

fig5_box.update_layout(
    yaxis_tickformat=".0%",
    legend_title_text="Categoría"
)

fig5_box.add_hline(y=0, line_dash="dash")

fig5_box.show()

In [ ]:
print("Conclusión del gráfico 5")
print("- La dispersión muestra que el descuento alto aumenta la probabilidad de margen negativo, pero no todos los productos reaccionan igual.")
print("- Esta visualización es útil para auditoría comercial: permite identificar productos concretos y operaciones de alto impacto.")
print("- Desde negocio, esto sugiere pasar de una política de descuento genérica a una política con límites por subcategoría y margen mínimo esperado.")

Conclusión del gráfico 5
- La dispersión muestra que el descuento alto aumenta la probabilidad de margen negativo, pero no todos los productos reaccionan igual.
- Esta visualización es útil para auditoría comercial: permite identificar productos concretos y operaciones de alto impacto.
- Desde negocio, esto sugiere pasar de una política de descuento genérica a una política con límites por subcategoría y margen mínimo esperado.


## 9. Propuesta final

Con los datos disponibles, la política de descuentos parece estar generando ventas a costa de rentabilidad. La conclusión debe formularse con precisión: **no se demuestra causalidad estricta**, pero sí una asociación empresarialmente relevante entre descuento e inferior margen.

### Recomendación operativa

1. **Establecer un suelo de margen por subcategoría.** No aprobar descuentos que dejen el margen esperado por debajo de cero salvo liquidación justificada.
2. **Revisar descuentos superiores al 20%-30%.** En este dataset, esos tramos entran en zona de pérdida agregada.
3. **Aplicar controles específicos a subcategorías problemáticas.** Prioridad para `Tables`, `Machines`, `Bookcases`, `Binders`, `Appliances`, `Storage` y `Furnishings` cuando se vendan con descuento.
4. **No eliminar todos los descuentos.** Algunas subcategorías pueden seguir siendo rentables con descuento; la política debe ser segmentada, no indiscriminada.
5. **Crear seguimiento periódico.** KPI mínimo mensual: ventas, beneficio, margen ponderado, descuento medio y beneficio por unidad por `Category` y `Sub-Category`.

### KPI recomendado para seguimiento

El KPI principal debe ser el **margen ponderado**:

$$
\text{Margen ponderado} = \frac{\sum Profit}{\sum Sales}
$$
\]

Es preferible al promedio simple de márgenes porque evita que muchas operaciones pequeñas distorsionen la lectura de rentabilidad agregada.

In [ ]:
# Tabla final de control propuesta: subcategorías con descuento y beneficio negativo
control_descuentos = (
    resumen_subcategoria
    .query("flag_descuento == 'Con Descuento' and beneficio < 0")
    .sort_values("beneficio")
    .assign(
        recomendacion=lambda x: np.where(
            x["margen_ponderado"] < -0.10,
            "Bloquear o justificar descuento",
            "Revisar límite de descuento"
        )
    )
    [["Category", "Sub-Category", "ventas", "beneficio", "margen_ponderado", "descuento_medio", "lineas", "recomendacion"]]
)

control_descuentos

,Category,Sub-Category,ventas,beneficio,margen_ponderado,descuento_medio,lineas,recomendacion
6,Furniture,Tables,"135,386.77","-31,001.78",-0.23,0.34,247,Bloquear o justificar descuento
30,Technology,Machines,"118,204.63","-23,753.07",-0.20,0.41,86,Bloquear o justificar descuento
0,Furniture,Bookcases,"82,944.02","-9,548.27",-0.12,0.29,168,Bloquear o justificar descuento
12,Office Supplies,Binders,"121,583.25","-9,092.69",-0.07,0.48,1186,Revisar límite de descuento
8,Office Supplies,Appliances,"29,465.97","-5,045.73",-0.17,0.40,195,Bloquear o justificar descuento
22,Office Supplies,Storage,"65,989.85","-4,249.35",-0.06,0.20,316,Revisar límite de descuento
4,Furniture,Furnishings,"30,255.36","-3,788.83",-0.13,0.34,386,Bloquear o justificar descuento
24,Office Supplies,Supplies,"15,114.33","-2,907.49",-0.19,0.20,73,Bloquear o justificar descuento


In [ ]:
# Resumen ejecutivo numérico reproducible
ventas_totales = df["Sales"].sum()
beneficio_total = df["Profit"].sum()
margen_total = beneficio_total / ventas_totales
porcentaje_lineas_descuento = (df["Discount"] > 0).mean()

print("Resumen ejecutivo")
print(f"- Ventas totales: {ventas_totales:,.2f}")
print(f"- Beneficio total: {beneficio_total:,.2f}")
print(f"- Margen ponderado total: {margen_total:.2%}")
print(f"- Porcentaje de líneas con descuento: {porcentaje_lineas_descuento:.2%}")
print(f"- Beneficio con descuento: {benef_con_desc:,.2f}")
print(f"- Beneficio sin descuento: {benef_sin_desc:,.2f}")
print("- Diagnóstico: la empresa no tiene un problema de ventas; tiene un problema de calidad de margen en operaciones descontadas.")

Resumen ejecutivo
- Ventas totales: 2,297,377.17
- Beneficio total: 286,455.96
- Margen ponderado total: 12.47%
- Porcentaje de líneas con descuento: 51.98%
- Beneficio con descuento: -34,590.58
- Beneficio sin descuento: 321,046.54
- Diagnóstico: la empresa no tiene un problema de ventas; tiene un problema de calidad de margen en operaciones descontadas.
